# V12 - Qualidade, governanca e performance

**Objetivo:** executar controles mensuraveis sobre uma amostra (completude, unicidade, rejeicoes), observar governanca via estatisticas do projeto e comparar custo de consulta ampla x limitada.

In [ ]:
# Carrega a configuracao local e cria o cliente CDF. / Load local configuration and create the CDF client.
import os
from getpass import getpass
from pathlib import Path

from dotenv import load_dotenv
from cognite.client import ClientConfig, CogniteClient
from cognite.client.credentials import OAuthClientCredentials, Token

env_file = Path('.env')
pkg_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / '.env.example').exists()),
    Path.cwd(),
)

if env_file.exists():
    load_dotenv(env_file)
else:
    for name in ('.env', '.env.example'):
        candidate = pkg_root / name
        if candidate.exists():
            load_dotenv(candidate)
            break

if not os.environ.get('CDF_CLUSTER'):
    os.environ['CDF_CLUSTER'] = input('CDF_CLUSTER: ').strip()
if not os.environ.get('CDF_PROJECT'):
    os.environ['CDF_PROJECT'] = input('CDF_PROJECT: ').strip()

base_url = os.environ.get('CDF_URL', '').rstrip('/') or f"https://{os.environ['CDF_CLUSTER']}.cognitedata.com"
oauth_ready = env_file.exists() and all(
    os.environ.get(key) for key in ('IDP_TOKEN_URL', 'IDP_CLIENT_ID', 'IDP_CLIENT_SECRET')
)
if oauth_ready:
    scopes = os.environ.get('IDP_SCOPES', '').replace(',', ' ').split() or [f'{base_url}/.default']
    audience = os.environ.get('IDP_AUDIENCE', '')
    credentials = OAuthClientCredentials(
        token_url=os.environ['IDP_TOKEN_URL'],
        client_id=os.environ['IDP_CLIENT_ID'],
        client_secret=os.environ['IDP_CLIENT_SECRET'],
        scopes=scopes,
        **({'audience': audience} if audience else {}),
    )
else:
    # Sem .env nesta pasta: usa Bearer Token.
    bearer_token = (os.environ.get('CDF_BEARER_TOKEN') or getpass('CDF_BEARER_TOKEN: ')).strip()
    if bearer_token.lower().startswith('bearer '):
        bearer_token = bearer_token[7:].strip()
    credentials = Token(bearer_token)

client = CogniteClient(ClientConfig(
    client_name='radix-cdf-onboarding-v12',
    base_url=base_url,
    project=os.environ['CDF_PROJECT'],
    credentials=credentials,
))

## 1. Preparar amostra com um defeito proposital
Um node tem `name` nulo para exercitar a checagem de completude.

In [ ]:
from uuid import uuid4
import pandas as pd
from cognite.client.data_classes.data_modeling import (
    ContainerApply, ContainerId, Float64, MappedPropertyApply, NodeApply, NodeId,
    NodeOrEdgeData, SpaceApply, Text, ViewApply, ViewId,
)

run = uuid4().hex[:8]
sp = f'sp_ur_training_v12_{run}'
container_id = ContainerId(sp, 'Reading')
view_id = ViewId(sp, 'Reading', 'v1')

client.data_modeling.spaces.apply(SpaceApply(space=sp, name='UR training - V12'))
client.data_modeling.containers.apply(ContainerApply._load({
    'space': sp, 'externalId': container_id.external_id,
    'properties': {
        'name': {'type': Text().dump(), 'nullable': True},
        'value': {'type': Float64().dump(), 'nullable': True},
    },
    'usedFor': 'node',
}))
client.data_modeling.views.apply(ViewApply(
    space=sp, external_id=view_id.external_id, version=view_id.version,
    properties={
        'name': MappedPropertyApply(container=container_id, container_property_identifier='name'),
        'value': MappedPropertyApply(container=container_id, container_property_identifier='value'),
    },
))
rows = [('r1', 'Leitura 1', 10.0), ('r2', 'Leitura 2', 20.0), ('r3', None, 30.0)]
client.data_modeling.instances.apply(nodes=[
    NodeApply(space=sp, external_id=xid,
              sources=[NodeOrEdgeData(source=view_id, properties={'name': nm, 'value': val})])
    for xid, nm, val in rows
])
print('amostra criada:', len(rows), 'nodes em', sp)

## 2. Controles de qualidade
Completude (taxa de nao-nulos), unicidade de chaves e contagem de rejeicoes.

In [ ]:
nodes = client.data_modeling.instances.list(instance_type='node', sources=view_id, space=sp, limit=100)
df = nodes.to_pandas()

total = len(df)
completeness = df['name'].notna().mean() if 'name' in df.columns else 0.0
keys = list(zip(df['space'], df['external_id'])) if 'external_id' in df.columns else []
uniqueness_ok = len(keys) == len(set(keys))
rejected = df[df['name'].isna()] if 'name' in df.columns else df.iloc[0:0]

metrics = {
    'total': total,
    'completude_name': round(float(completeness), 3),
    'unicidade_ok': uniqueness_ok,
    'rejeicoes': len(rejected),
}
print('controles de qualidade:', metrics)
rejected

## 3. Governanca: estatisticas do projeto
Visao agregada de uso do DMS (protegida por try/except).

In [ ]:
from cognite.client.exceptions import CogniteAPIError

try:
    stats = client.data_modeling.statistics.project()
    print('estatisticas do projeto:')
    print(stats.dump())
except CogniteAPIError as exc:
    print('Sem acesso as estatisticas do projeto neste ambiente:', exc.code)

## 4. Performance: consulta ampla x limitada
Mede a latencia de listar com limites diferentes.

In [ ]:
import time
import matplotlib.pyplot as plt

limits = [1, 10, 100]
timings = []
for lim in limits:
    t0 = time.perf_counter()
    client.data_modeling.instances.list(instance_type='node', limit=lim)
    timings.append((time.perf_counter() - t0) * 1000)

for lim, ms in zip(limits, timings):
    print(f'limit={lim:>4}: {ms:7.1f} ms')

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([str(l) for l in limits], timings)
ax.set_title('Latencia por tamanho de pagina')
ax.set_xlabel('limit')
ax.set_ylabel('ms')
plt.tight_layout()
plt.show()

## Mini-exercicio
- Adicione a checagem de completude para `value` e reporte as duas taxas juntas.
- Repita a medicao de performance com `sources=view_id` e compare com a consulta ampla.

## 5. Limpeza idempotente

In [ ]:
assert sp.startswith('sp_ur_training_v12_')
client.data_modeling.instances.delete(nodes=[NodeId(sp, xid) for xid, *_ in rows])
client.data_modeling.views.delete(view_id)
client.data_modeling.containers.delete(container_id)
client.data_modeling.spaces.delete(sp)
print('space_still_exists:', client.data_modeling.spaces.retrieve(sp) is not None)